In [1]:
# ================================================================
# TB PORTALS MARCH 2025
# NOTEBOOK 04
# FINAL DR-TB vs DS-TB COHORT CONSTRUCTION
# ================================================================
#
# PROJECT:
# Multimodal Tuberculosis Drug-Resistance Prediction Using
# Chest X-Ray Images and Mycobacterium tuberculosis
# Genomic Features
#
# OBJECTIVE OF THIS NOTEBOOK
# --------------------------
# Construct and audit the final condition-level candidate cohort
# for DR-TB vs DS-TB multimodal prediction.
#
# THIS NOTEBOOK DOES NOT:
# -----------------------
# - preprocess DICOM images
# - resize images
# - perform CLAHE
# - perform lung segmentation
# - engineer genomic ML features
# - train any model
# - perform XAI
#
# It only establishes the scientifically auditable cohort.
#
# CASE UNIT:
# condition_id
#
# ================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")


# ================================================================
# 1. PROJECT PATHS
# ================================================================

BASE_DIR = Path(r"C:\TBP")
METADATA_DIR = BASE_DIR / "Metadata"


CXR_SOURCE = (
    METADATA_DIR /
    "TB_Portals_CXRs_March_2025.csv"
)

GENOMIC_SOURCE = (
    METADATA_DIR /
    "TB_Portals_Genomics_March_2025.csv"
)

CXR_MASTER = (
    METADATA_DIR /
    "Step_3B_6_24_Notebook_03" /
    "TB_Portals_March2025_3859_Multimodal_CXR_Master.csv"
)

TEMPORAL_PAIRS = (
    METADATA_DIR /
    "Step_3B_6_34_CXR_Genomics_Temporal_Reconciliation" /
    "TB_Portals_March2025_Step_3B_6_34_All_CXR_Genomic_Temporal_Pairs.csv"
)

GENOMIC_TIMEPOINTS = (
    METADATA_DIR /
    "Step_3B_6_34_CXR_Genomics_Temporal_Reconciliation" /
    "TB_Portals_March2025_Step_3B_6_34_Unique_Genomic_Timepoints_Common_Cohort.csv"
)


# ================================================================
# 2. OUTPUT DIRECTORY
# ================================================================

OUTPUT_DIR = (
    METADATA_DIR /
    "Step_3B_6_49_Final_DR_DS_Cohort"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. START
# ================================================================

print("=" * 100)
print("TB PORTALS MARCH 2025")
print("NOTEBOOK 04 — FINAL DR-TB vs DS-TB COHORT CONSTRUCTION")
print("=" * 100)


# ================================================================
# 4. INPUT VALIDATION
# ================================================================

print("\n1. INPUT FILE VALIDATION")
print("-" * 100)


required_inputs = {
    "CXR source": CXR_SOURCE,
    "Genomic source": GENOMIC_SOURCE,
    "CXR master": CXR_MASTER,
    "Temporal pairs": TEMPORAL_PAIRS,
    "Genomic timepoints": GENOMIC_TIMEPOINTS
}


for name, path in required_inputs.items():

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired input not found:\n"
            f"{name}\n"
            f"{path}"
        )

    print(
        f"{name:<30}: FOUND"
    )


# ================================================================
# 5. LOAD DATA
# ================================================================

print("\n2. LOADING AUTHORITATIVE DATA")
print("-" * 100)


cxr = pd.read_csv(
    CXR_SOURCE,
    low_memory=False
)

genomics = pd.read_csv(
    GENOMIC_SOURCE,
    low_memory=False
)

cxr_master = pd.read_csv(
    CXR_MASTER,
    low_memory=False
)

temporal = pd.read_csv(
    TEMPORAL_PAIRS,
    low_memory=False
)

genomic_timepoints = pd.read_csv(
    GENOMIC_TIMEPOINTS,
    low_memory=False
)


print(
    f"CXR source rows             : {len(cxr):,}"
)

print(
    f"Genomic source rows         : {len(genomics):,}"
)

print(
    f"CXR master rows             : {len(cxr_master):,}"
)

print(
    f"Temporal pair rows          : {len(temporal):,}"
)

print(
    f"Unique genomic timepoints   : {len(genomic_timepoints):,}"
)


# ================================================================
# 6. REQUIRED COLUMN DEFINITIONS
# ================================================================

print("\n3. SCHEMA VALIDATION")
print("-" * 100)


required_cxr = [
    "condition_id",
    "patient_id",
    "imagingstudy_id",
    "imaging_date",
    "series_instance_content_url",
    "type_of_resistance",
    "outcome",
    "case_definition",
    "diagnosis_code"
]


required_genomic = [
    "condition_id",
    "type_of_resistance",
    "drug_resistance_type"
]


required_temporal = [
    "condition_id"
]


def validate_columns(
    dataframe,
    required,
    name
):

    missing = [
        c
        for c in required
        if c not in dataframe.columns
    ]

    if missing:

        raise ValueError(
            f"{name} is missing required columns:\n"
            f"{missing}"
        )

    print(
        f"{name:<30}: PASS"
    )


validate_columns(
    cxr,
    required_cxr,
    "CXR schema"
)

validate_columns(
    genomics,
    required_genomic,
    "Genomic schema"
)

validate_columns(
    cxr_master,
    [
        "condition_id",
        "series_instance_content_url"
    ],
    "CXR master schema"
)

validate_columns(
    temporal,
    required_temporal,
    "Temporal schema"
)


# ================================================================
# 7. CONDITION-ID NORMALIZATION
# ================================================================

print("\n4. CONDITION-ID NORMALIZATION")
print("-" * 100)


def normalize_condition_id(
    dataframe
):

    dataframe = dataframe.copy()

    dataframe["condition_id"] = (
        dataframe["condition_id"]
        .astype("string")
        .str.strip()
    )

    return dataframe


cxr = normalize_condition_id(cxr)
genomics = normalize_condition_id(genomics)
cxr_master = normalize_condition_id(cxr_master)
temporal = normalize_condition_id(temporal)
genomic_timepoints = normalize_condition_id(
    genomic_timepoints
)


for name, dataframe in [
    ("CXR", cxr),
    ("Genomics", genomics),
    ("CXR master", cxr_master),
    ("Temporal", temporal),
    ("Genomic timepoints", genomic_timepoints)
]:

    if dataframe["condition_id"].isna().any():

        raise ValueError(
            f"{name} contains missing condition_id."
        )


print(
    "Condition-ID normalization : PASS"
)


# ================================================================
# 8. CORE MULTIMODAL POPULATION
# ================================================================

print("\n5. CORE MULTIMODAL POPULATION")
print("-" * 100)


cxr_conditions = set(
    cxr_master[
        "condition_id"
    ]
    .dropna()
    .unique()
)

genomic_conditions = set(
    genomics[
        "condition_id"
    ]
    .dropna()
    .unique()
)


common_conditions = (
    cxr_conditions
    &
    genomic_conditions
)


print(
    f"CXR master conditions      : "
    f"{len(cxr_conditions):,}"
)

print(
    f"Genomic conditions         : "
    f"{len(genomic_conditions):,}"
)

print(
    f"Common conditions          : "
    f"{len(common_conditions):,}"
)


if len(cxr_conditions) != 3081:

    raise ValueError(
        "Expected 3,081 CXR multimodal conditions."
    )

if len(common_conditions) != 3081:

    raise ValueError(
        "Expected 3,081 CXR+genomic common conditions."
    )


# ================================================================
# 9. CXR CONDITION-LEVEL LABEL MASTER
# ================================================================

print("\n6. CXR CONDITION-LEVEL LABEL MASTER")
print("-" * 100)


cxr_common = cxr[
    cxr["condition_id"].isin(
        common_conditions
    )
].copy()


def clean_label(series):

    return (
        series
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA
        })
    )


for col in [
    "type_of_resistance",
    "outcome",
    "case_definition",
    "diagnosis_code"
]:

    cxr_common[col] = clean_label(
        cxr_common[col]
    )


# ------------------------------------------------
# Verify resistance stability
# ------------------------------------------------

cxr_resistance_counts = (
    cxr_common
    .groupby("condition_id")[
        "type_of_resistance"
    ]
    .nunique(dropna=True)
)


cxr_resistance_multiple = (
    cxr_resistance_counts
    > 1
)


print(
    "Conditions with multiple CXR resistance labels :",
    int(cxr_resistance_multiple.sum())
)


if cxr_resistance_multiple.any():

    raise ValueError(
        "CXR resistance labels are not stable at "
        "condition level."
    )


# ------------------------------------------------
# Build condition-level CXR label
# ------------------------------------------------

cxr_label_master = (
    cxr_common
    .groupby("condition_id")
    .agg(
        cxr_resistance=(
            "type_of_resistance",
            lambda x: x.dropna().iloc[0]
            if x.dropna().shape[0] > 0
            else pd.NA
        ),

        cxr_outcome=(
            "outcome",
            lambda x: x.dropna().iloc[0]
            if x.dropna().shape[0] > 0
            else pd.NA
        ),

        cxr_case_definition=(
            "case_definition",
            lambda x: x.dropna().iloc[0]
            if x.dropna().shape[0] > 0
            else pd.NA
        ),

        cxr_diagnosis_code=(
            "diagnosis_code",
            lambda x: x.dropna().iloc[0]
            if x.dropna().shape[0] > 0
            else pd.NA
        ),

        cxr_record_count=(
            "condition_id",
            "size"
        ),

        cxr_url_count=(
            "series_instance_content_url",
            "nunique"
        ),

        patient_id_count=(
            "patient_id",
            "nunique"
        )
    )
    .reset_index()
)


if len(cxr_label_master) != 3081:

    raise ValueError(
        "Condition-level CXR master does not contain "
        "exactly 3,081 conditions."
    )


# ================================================================
# 10. GENOMIC CONDITION-LEVEL LABEL MASTER
# ================================================================

print("\n7. GENOMIC CONDITION-LEVEL LABEL MASTER")
print("-" * 100)


genomic_common = genomics[
    genomics["condition_id"].isin(
        common_conditions
    )
].copy()


for col in [
    "type_of_resistance",
    "drug_resistance_type"
]:

    genomic_common[col] = clean_label(
        genomic_common[col]
    )


# ------------------------------------------------
# Genomic type_of_resistance stability
# ------------------------------------------------

genomic_resistance_counts = (
    genomic_common
    .groupby("condition_id")[
        "type_of_resistance"
    ]
    .nunique(dropna=True)
)


if (
    genomic_resistance_counts > 1
).any():

    raise ValueError(
        "Genomic type_of_resistance is not stable "
        "at condition level."
    )


# ------------------------------------------------
# Build genomic label master
# ------------------------------------------------

def first_non_missing(series):

    values = series.dropna()

    if len(values) == 0:

        return pd.NA

    return values.iloc[0]


genomic_label_master = (
    genomic_common
    .groupby("condition_id")
    .agg(
        genomic_resistance=(
            "type_of_resistance",
            first_non_missing
        ),

        genomic_drug_resistance_values=(
            "drug_resistance_type",
            lambda x:
                sorted(
                    set(
                        x.dropna()
                        .astype(str)
                    )
                )
        ),

        genomic_record_count=(
            "condition_id",
            "size"
        )
    )
    .reset_index()
)


genomic_label_master[
    "genomic_drug_resistance_nunique"
] = (
    genomic_label_master[
        "genomic_drug_resistance_values"
    ]
    .apply(len)
)


# ================================================================
# 11. CXR-GENOMIC RESISTANCE AGREEMENT
# ================================================================

print("\n8. CXR ↔ GENOMIC RESISTANCE AGREEMENT")
print("-" * 100)


labels = (
    cxr_label_master[
        [
            "condition_id",
            "cxr_resistance"
        ]
    ]
    .merge(
        genomic_label_master[
            [
                "condition_id",
                "genomic_resistance"
            ]
        ],
        on="condition_id",
        how="inner",
        validate="one_to_one"
    )
)


labels[
    "resistance_agreement"
] = (
    labels["cxr_resistance"]
    ==
    labels["genomic_resistance"]
)


agreement_count = int(
    labels[
        "resistance_agreement"
    ].sum()
)

disagreement_count = (
    len(labels)
    -
    agreement_count
)


print(
    f"Agreement                  : "
    f"{agreement_count:,}"
)

print(
    f"Disagreement               : "
    f"{disagreement_count:,}"
)


if disagreement_count != 0:

    disagreement_file = (
        OUTPUT_DIR /
        "TB_Portals_March2025_Step_3B_6_49_Resistance_Label_Disagreements.csv"
    )

    labels[
        ~labels["resistance_agreement"]
    ].to_csv(
        disagreement_file,
        index=False,
        encoding="utf-8-sig"
    )

    raise ValueError(
        "CXR/genomic resistance labels disagree. "
        "Review the saved disagreement file before proceeding."
    )


# ================================================================
# 12. TARGET CATEGORIES
# ================================================================

print("\n9. RESISTANCE CATEGORY DISTRIBUTION")
print("-" * 100)


resistance_distribution = (
    labels[
        "cxr_resistance"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "resistance_category"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    resistance_distribution.to_string(
        index=False
    )
)


# ================================================================
# 13. TARGET DEFINITION
# ================================================================
#
# For this project:
#
# DS-TB:
#     Sensitive
#
# DR-TB:
#     All explicitly classified non-Sensitive resistance
#     categories present in the common cohort.
#
# IMPORTANT:
# This is the binary target definition to be audited.
# It does NOT use genomic_drug_resistance_type.
#
# ================================================================

print("\n10. BINARY DR-TB / DS-TB TARGET DEFINITION")
print("-" * 100)


DS_LABEL = "Sensitive"


known_categories = set(
    labels[
        "cxr_resistance"
    ]
    .dropna()
    .unique()
)


if DS_LABEL not in known_categories:

    raise ValueError(
        "Sensitive category is absent from the common cohort."
    )


labels[
    "target_class"
] = np.where(
    labels[
        "cxr_resistance"
    ]
    == DS_LABEL,
    "DS-TB",
    "DR-TB"
)


labels[
    "target_binary"
] = np.where(
    labels[
        "target_class"
    ]
    == "DS-TB",
    0,
    1
)


target_distribution = (
    labels[
        "target_class"
    ]
    .value_counts()
    .rename_axis(
        "target_class"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    target_distribution.to_string(
        index=False
    )
)


# ================================================================
# 14. TARGET CATEGORY BREAKDOWN WITHIN BINARY CLASSES
# ================================================================

print("\n11. DR-TB INTERNAL CATEGORY BREAKDOWN")
print("-" * 100)


dr_breakdown = (
    labels[
        labels["target_class"] == "DR-TB"
    ][
        "cxr_resistance"
    ]
    .value_counts()
    .rename_axis(
        "resistance_category"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    dr_breakdown.to_string(
        index=False
    )
)


# ================================================================
# 15. GENOMIC DRUG-RESISTANCE AMBIGUITY
# ================================================================

print("\n12. GENOMIC DRUG-RESISTANCE PROFILE AMBIGUITY")
print("-" * 100)


genomic_ambiguity = genomic_label_master[
    genomic_label_master[
        "genomic_drug_resistance_nunique"
    ] > 1
].copy()


print(
    f"Conditions with multiple genomic "
    f"drug_resistance_type values: "
    f"{len(genomic_ambiguity):,}"
)


# ================================================================
# 16. CXR IMAGE MULTIPLICITY
# ================================================================

print("\n13. CXR IMAGE MULTIPLICITY")
print("-" * 100)


cxr_label_master[
    "multiple_cxr"
] = (
    cxr_label_master[
        "cxr_url_count"
    ]
    > 1
)


multiple_cxr_summary = (
    cxr_label_master[
        "cxr_url_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "number_of_cxr_images"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    multiple_cxr_summary.to_string(
        index=False
    )
)


print(
    f"Conditions with >1 CXR     : "
    f"{int(cxr_label_master['multiple_cxr'].sum()):,}"
)


# ================================================================
# 17. TEMPORAL STRUCTURE
# ================================================================

print("\n14. CXR ↔ GENOMIC TEMPORAL STRUCTURE")
print("-" * 100)


# ------------------------------------------------
# Identify available temporal columns
# ------------------------------------------------

temporal_columns = [
    c
    for c in [
        "condition_id",
        "imaging_date",
        "specimen_collection_date_num",
        "signed_delta_days",
        "temporal_direction"
    ]
    if c in temporal.columns
]


print(
    "Available temporal columns:"
)

for col in temporal_columns:

    print(
        f"  - {col}"
    )


if "temporal_direction" in temporal.columns:

    temporal_distribution = (
        temporal[
            temporal["condition_id"]
            .isin(common_conditions)
        ]
        .groupby(
            "temporal_direction"
        )
        .agg(
            pair_count=(
                "condition_id",
                "size"
            ),

            condition_count=(
                "condition_id",
                "nunique"
            )
        )
        .reset_index()
    )

    print(
        temporal_distribution.to_string(
            index=False
        )
    )

else:

    print(
        "Temporal direction column not found; "
        "temporal direction will be reconstructed in "
        "the dedicated temporal audit."
    )


# ================================================================
# 18. TEMPORAL ELIGIBILITY — DO NOT AUTOMATICALLY EXCLUDE YET
# ================================================================
#
# We deliberately do NOT impose:
#
#     before-only
#     same-day-only
#     after-only
#
# in this notebook without auditing the consequences.
#
# Instead we calculate availability for each condition.
#
# ================================================================

if (
    "signed_delta_days" in temporal.columns
):

    temporal_condition_summary = (
        temporal[
            temporal["condition_id"]
            .isin(common_conditions)
        ]
        .groupby("condition_id")
        .agg(
            temporal_pair_count=(
                "condition_id",
                "size"
            ),

            min_signed_delta_days=(
                "signed_delta_days",
                "min"
            ),

            max_signed_delta_days=(
                "signed_delta_days",
                "max"
            )
        )
        .reset_index()
    )

else:

    temporal_condition_summary = (
        temporal[
            [
                "condition_id"
            ]
        ]
        .drop_duplicates()
        .assign(
            temporal_pair_count=1
        )
    )


# ================================================================
# 19. MERGE CONDITION-LEVEL MASTER
# ================================================================

print("\n15. BUILDING CONDITION-LEVEL MASTER")
print("-" * 100)


condition_master = (
    labels
    .merge(
        genomic_label_master[
            [
                "condition_id",
                "genomic_record_count",
                "genomic_drug_resistance_nunique"
            ]
        ],
        on="condition_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        cxr_label_master[
            [
                "condition_id",
                "cxr_record_count",
                "cxr_url_count",
                "multiple_cxr",
                "patient_id_count"
            ]
        ],
        on="condition_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        temporal_condition_summary,
        on="condition_id",
        how="left",
        validate="one_to_one"
    )
)


if len(condition_master) != 3081:

    raise ValueError(
        "Final condition master does not contain "
        "exactly 3,081 conditions."
    )


# ================================================================
# 20. DATA COMPLETENESS AUDIT
# ================================================================

print("\n16. DATA COMPLETENESS AUDIT")
print("-" * 100)


completeness_checks = {

    "Target class missing":
        int(
            condition_master[
                "target_class"
            ].isna().sum()
        ),

    "CXR record count missing":
        int(
            condition_master[
                "cxr_record_count"
            ].isna().sum()
        ),

    "CXR URL count missing":
        int(
            condition_master[
                "cxr_url_count"
            ].isna().sum()
        ),

    "Genomic record count missing":
        int(
            condition_master[
                "genomic_record_count"
            ].isna().sum()
        )
}


for name, count in completeness_checks.items():

    print(
        f"{name:<40}: {count}"
    )


if any(
    count != 0
    for count in completeness_checks.values()
):

    raise ValueError(
        "Condition-level completeness failure."
    )


# ================================================================
# 21. MULTIMODAL CLASS AVAILABILITY
# ================================================================

print("\n17. FINAL MULTIMODAL CLASS AVAILABILITY")
print("-" * 100)


class_availability = (
    condition_master
    .groupby(
        "target_class"
    )
    .agg(

        conditions=(
            "condition_id",
            "nunique"
        ),

        cxr_records=(
            "cxr_record_count",
            "sum"
        ),

        genomic_records=(
            "genomic_record_count",
            "sum"
        ),

        conditions_with_multiple_cxr=(
            "multiple_cxr",
            "sum"
        ),

        conditions_with_multiple_genomic_values=(
            "genomic_drug_resistance_nunique",
            lambda x:
                int(
                    (x > 1).sum()
                )
        )
    )
    .reset_index()
)


class_availability[
    "condition_percentage"
] = (
    class_availability[
        "conditions"
    ]
    /
    class_availability[
        "conditions"
    ].sum()
    *
    100
)


print(
    class_availability.to_string(
        index=False
    )
)


# ================================================================
# 22. TARGET BALANCE
# ================================================================

print("\n18. TARGET BALANCE")
print("-" * 100)


target_counts = (
    condition_master[
        "target_class"
    ]
    .value_counts()
)


total_conditions = len(
    condition_master
)


for target_class, count in target_counts.items():

    percentage = (
        count /
        total_conditions *
        100
    )

    print(
        f"{target_class:<10}: "
        f"{count:>6,} "
        f"({percentage:.2f}%)"
    )


# ================================================================
# 23. NO AUTOMATIC CXR EXCLUSION
# ================================================================

print("\n19. IMAGE-ELIGIBILITY SAFETY")
print("-" * 100)

print(
    "No new automatic image exclusion is applied."
)

print(
    "Previous semantic/structural review artifacts remain "
    "evidence for the subsequent CXR eligibility stage."
)


# ================================================================
# 24. NO AUTOMATIC TEMPORAL EXCLUSION
# ================================================================

print(
    "\n20. TEMPORAL-ELIGIBILITY SAFETY"
)

print(
    "No before/same-day/after temporal category is "
    "automatically excluded in this first target audit."
)


# ================================================================
# 25. NO TARGET LEAKAGE FEATURES
# ================================================================

print(
    "\n21. TARGET-LEAKAGE LOCK"
)

target_leakage_fields = [
    "type_of_resistance",
    "drug_resistance_type"
]


for field in target_leakage_fields:

    print(
        f"{field:<30}: TARGET-LIKE — NOT A MODEL FEATURE"
    )


# ================================================================
# 26. FINAL COHORT STATUS
# ================================================================

print(
    "\n22. NOTEBOOK 04 COHORT STATUS"
)
print("-" * 100)


final_candidate_count = len(
    condition_master
)

final_dr_count = int(
    (
        condition_master[
            "target_class"
        ]
        ==
        "DR-TB"
    ).sum()
)

final_ds_count = int(
    (
        condition_master[
            "target_class"
        ]
        ==
        "DS-TB"
    ).sum()
)


print(
    f"Multimodal candidate conditions : "
    f"{final_candidate_count:,}"
)

print(
    f"DR-TB conditions                : "
    f"{final_dr_count:,}"
)

print(
    f"DS-TB conditions                : "
    f"{final_ds_count:,}"
)

print(
    f"DR + DS total                   : "
    f"{final_dr_count + final_ds_count:,}"
)


if (
    final_dr_count +
    final_ds_count
    !=
    3081
):

    raise ValueError(
        "DR + DS counts do not reconcile to "
        "the 3,081 multimodal conditions."
    )


# ================================================================
# 27. SAVE CONDITION MASTER
# ================================================================

print(
    "\n23. SAVING OUTPUTS"
)
print("-" * 100)


condition_master_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Final_Target_Candidate_Condition_Master.csv"
)

condition_master.to_csv(
    condition_master_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"{condition_master_path.name}: PASS"
)


# ================================================================
# 28. SAVE TARGET DISTRIBUTION
# ================================================================

target_distribution_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Target_Distribution.csv"
)

target_distribution.to_csv(
    target_distribution_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"{target_distribution_path.name}: PASS"
)


# ================================================================
# 29. SAVE DR BREAKDOWN
# ================================================================

dr_breakdown_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_DR_Category_Breakdown.csv"
)

dr_breakdown.to_csv(
    dr_breakdown_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"{dr_breakdown_path.name}: PASS"
)


# ================================================================
# 30. SAVE CLASS AVAILABILITY
# ================================================================

class_availability_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Class_Availability.csv"
)

class_availability.to_csv(
    class_availability_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"{class_availability_path.name}: PASS"
)


# ================================================================
# 31. SAVE TEMPORAL SUMMARY
# ================================================================

temporal_summary_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Temporal_Condition_Summary.csv"
)

temporal_condition_summary.to_csv(
    temporal_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"{temporal_summary_path.name}: PASS"
)


# ================================================================
# 32. SAVE GENOMIC AMBIGUITY
# ================================================================

genomic_ambiguity_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Genomic_Drug_Resistance_Ambiguity.csv"
)

genomic_ambiguity.to_csv(
    genomic_ambiguity_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"{genomic_ambiguity_path.name}: PASS"
)


# ================================================================
# 33. SAVE FINAL DECISION RECORD
# ================================================================

decision_record = {

    "project_title":
        "Multimodal Tuberculosis Drug-Resistance Prediction "
        "Using Chest X-Ray Images and Mycobacterium tuberculosis "
        "Genomic Features",

    "case_unit":
        "condition_id",

    "candidate_multimodal_conditions":
        final_candidate_count,

    "descriptive_target_definition":
        "DS-TB = Sensitive; DR-TB = all other explicitly "
        "classified resistance categories",

    "dr_tb_candidate_conditions":
        final_dr_count,

    "ds_tb_candidate_conditions":
        final_ds_count,

    "target_source":
        "CXR type_of_resistance with genomic resistance "
        "agreement verification",

    "genomic_drug_resistance_type":
        "Target-like field; prohibited as direct model input",

    "temporal_rule":
        "NOT YET LOCKED",

    "representative_cxr_rule":
        "NOT YET LOCKED",

    "representative_genomic_specimen_rule":
        "NOT YET LOCKED",

    "final_image_eligibility":
        "NOT YET LOCKED",

    "final_modeling_cohort":
        "NOT YET LOCKED",

    "preprocessing_started":
        False,

    "lung_segmentation_started":
        False,

    "model_training_started":
        False,

    "xai_started":
        False,

    "raw_dicom_modified":
        False,

    "source_metadata_modified":
        False
}


decision_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Decision_Record.json"
)

with open(
    decision_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        decision_record,
        f,
        indent=4
    )

print(
    f"{decision_path.name}: PASS"
)


# ================================================================
# 34. FINAL VALIDATION
# ================================================================

print(
    "\n24. FINAL VALIDATION"
)
print("-" * 100)


validation = {

    "CXR source = 16,723":
        len(cxr) == 16723,

    "Genomic source = 4,467":
        len(genomics) == 4467,

    "CXR master = 3,859":
        len(cxr_master) == 3859,

    "Multimodal conditions = 3,081":
        len(common_conditions) == 3081,

    "CXR/genomic resistance agreement = 3,081":
        agreement_count == 3081,

    "DR + DS = 3,081":
        (
            final_dr_count +
            final_ds_count
            ==
            3081
        ),

    "No preprocessing":
        True,

    "No segmentation":
        True,

    "No model training":
        True,

    "No XAI":
        True,

    "Raw DICOM unchanged":
        True,

    "Source metadata unchanged":
        True
}


all_pass = True

for check, result in validation.items():

    print(
        f"{check:<55}: "
        f"{'PASS' if result else 'FAIL'}"
    )

    if not result:

        all_pass = False


if not all_pass:

    raise RuntimeError(
        "Notebook 04 final validation FAILED. "
        "Do not proceed to Notebook 05."
    )


# ================================================================
# 35. FINAL STATUS
# ================================================================

print()
print("=" * 100)
print("NOTEBOOK 04 — FINAL STATUS")
print("=" * 100)

print(
    f"Multimodal candidate conditions : "
    f"{final_candidate_count:,}"
)

print(
    f"DR-TB candidate conditions      : "
    f"{final_dr_count:,}"
)

print(
    f"DS-TB candidate conditions      : "
    f"{final_ds_count:,}"
)

print(
    "Temporal rule                   : NOT YET LOCKED"
)

print(
    "Representative CXR rule        : NOT YET LOCKED"
)

print(
    "Representative genomic rule    : NOT YET LOCKED"
)

print(
    "Final image eligibility         : NOT YET LOCKED"
)

print(
    "Final modeling cohort           : NOT YET LOCKED"
)

print(
    "Notebook 05 authorization       : NOT YET AUTHORIZED"
)

print(
    "Technical validation             : PASS"
)

print("=" * 100)
print("NOTEBOOK 04 COMPLETE")
print("=" * 100)

TB PORTALS MARCH 2025
NOTEBOOK 04 — FINAL DR-TB vs DS-TB COHORT CONSTRUCTION

1. INPUT FILE VALIDATION
----------------------------------------------------------------------------------------------------
CXR source                    : FOUND
Genomic source                : FOUND
CXR master                    : FOUND
Temporal pairs                : FOUND
Genomic timepoints            : FOUND

2. LOADING AUTHORITATIVE DATA
----------------------------------------------------------------------------------------------------
CXR source rows             : 16,723
Genomic source rows         : 4,467
CXR master rows             : 3,859
Temporal pair rows          : 4,146
Unique genomic timepoints   : 3,260

3. SCHEMA VALIDATION
----------------------------------------------------------------------------------------------------
CXR schema                    : PASS
Genomic schema                : PASS
CXR master schema             : PASS
Temporal schema               : PASS

4. CONDITION-ID NORMA